In [1]:
!pip install --upgrade huggingface_hub

from huggingface_hub import login
login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 109.2 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.4.2
    Uninstalling hf-xet-1.4.2:
      Successfully uninstalled hf-xet-1.4.2
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get('HF_TOKEN')
login(token=token)

In [4]:
!pip install transformers datasets evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=e4468982a6e5c67c17aa9dfff84746fae8c453e4c7684e57b1e093f0ecc6aa5c
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [5]:
from datasets import load_dataset

# Load the California portion of the BillSum dataset
dataset = load_dataset("billsum", split="ca_test")

# Look at the first legal text and its summary
print(f"Legal Text: {dataset[0]['text'][:200]}...")
print(f"Official Summary: {dataset[0]['summary']}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/91.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

data/ca_test-00000-of-00001.parquet:   0%|          | 0.00/6.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

Legal Text: The people of the State of California do enact as follows:


SECTION 1.
The Legislature finds and declares all of the following:
(a) (1) Since 1899 congressionally chartered veterans’ organizations ha...
Official Summary: Existing property tax law establishes a veterans’ organization exemption under which property is exempt from taxation if, among other things, that property is used exclusively for charitable purposes and is owned by a veterans’ organization.
This bill would provide that the veterans’ organization exemption shall not be denied to a property on the basis that the property is used for fraternal, lodge, or social club purposes, and would make specific findings and declarations in that regard. The bill would also provide that the exemption shall not apply to any portion of a property that consists of a bar where alcoholic beverages are served.
Section 2229 of the Revenue and Taxation Code requires the Legislature to reimburse local agencies annually for certain

In [21]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Define the model name
model_name = "facebook/bart-large-cnn"

# Load the tokenizer and model, ensuring the model is moved to the specified device
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# 1. Access the specific string (the first legal document)
# Using [0] matches your working print statement
single_document = dataset[0]['text']

# 2. Tokenize the single string
inputs = tokenizer(
    single_document,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
).to(device) # Ensure inputs are on the device

# 3. Generate the summary
summary_ids = model.generate(
    inputs["input_ids"],
    num_beams=4,
    min_length=50,
    max_length=150,
    early_stopping=True
)

# 4. Decode the result back to English
summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("--- AI Generated Legal Summary ---")
print(summary_text)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

--- AI Generated Legal Summary ---
Since 1899 congressionally chartered veterans’ organizations have provided a valuable service to our nation’s returning service members. These organizations also own and manage various properties including lodges, posts, and fraternal halls. These properties act as a safe haven where veterans of all ages and their families can gather together.
